# ITBI data workflow

Implementation lives in `itbi_sao_paulo`. Run these cells in order to download, import, geocode, and export transactions.

In [1]:
from pathlib import Path

from itbi_sao_paulo import (
    GeocodingClient,
    enrich_transactions,
    load_or_fetch_monthly_indicators,
    GeocodingConfig,
    download_workbooks,
    load_transactions,
    select_transactions,
)

In [2]:
DATA_DIR = Path("_data")
ITBI_DIR = DATA_DIR / "itbi"
CACHE_DIR = DATA_DIR / "cache"
YEARS = range(2006, 2027)
OUTPUT_PATH = DATA_DIR / "data.csv"

## Download source workbooks

Annual workbooks are stored in `_data/itbi`. Cached BACEN data and the monthly indicators are stored in `_data/cache`; the final export is `_data/data.csv`.

In [3]:
download_summary = download_workbooks(YEARS, ITBI_DIR)

In [4]:
download_summary

,year,file,status,error
0,2006,_data/itbi/2006.xlsx,cached,None
1,2007,_data/itbi/2007.xlsx,cached,None
2,2008,_data/itbi/2008.xlsx,cached,None
3,2009,_data/itbi/2009.xlsx,cached,None
4,2010,_data/itbi/2010.xlsx,cached,None
5,2011,_data/itbi/2011.xlsx,cached,None
6,2012,_data/itbi/2012.xlsx,cached,None
7,2013,_data/itbi/2013.xlsx,cached,None
8,2014,_data/itbi/2014.xlsx,cached,None
9,2015,_data/itbi/2015.xlsx,cached,None


## Import monthly transactions

Internal column names use English. Source identifiers are imported as nullable strings, preserving leading zeros.

In [5]:
imported = load_transactions(ITBI_DIR)
df = imported.data
import_summary = imported.summary
extra_values = imported.extra_values

Reading workbooks:   0%|          | 0/21 [00:00<?, ?it/s]

Worksheets in 2006.xlsx:   0%|          | 0/16 [00:00<?, ?it/s]

Worksheets in 2007.xlsx:   0%|          | 0/16 [00:00<?, ?it/s]

Worksheets in 2008.xlsx:   0%|          | 0/16 [00:00<?, ?it/s]

Worksheets in 2009.xlsx:   0%|          | 0/16 [00:00<?, ?it/s]

Worksheets in 2010.xlsx:   0%|          | 0/16 [00:00<?, ?it/s]

... (12 outputs left out)

Worksheets in 2023.xlsx:   0%|          | 0/16 [00:00<?, ?it/s]

Worksheets in 2024.xlsx:   0%|          | 0/16 [00:00<?, ?it/s]

Worksheets in 2025.xlsx:   0%|          | 0/16 [00:00<?, ?it/s]

Worksheets in 2026.xlsx:   0%|          | 0/12 [00:00<?, ?it/s]

In [6]:
import_summary

,file,sheet,records
0,_data/itbi/2006.xlsx,JAN-2006,6710
1,_data/itbi/2006.xlsx,FEV-2006,7313
2,_data/itbi/2006.xlsx,MAR-2006,10108
3,_data/itbi/2006.xlsx,ABR-2006,8431
4,_data/itbi/2006.xlsx,MAI-2006,9531
...,...,...,...
243,_data/itbi/2026.xlsx,ABR-2026,20597
244,_data/itbi/2026.xlsx,MAI-2026,21283
245,_data/itbi/2026.xlsx,JUN-2026,22051
246,_data/itbi/2026.xlsx,JUL-2026,22560


In [7]:
df.head()

,property_tax_id,street_name,street_number,address_complement,neighborhood,address_reference,postal_code,transaction_nature,declared_transaction_value,transaction_date,...,property_tax_status,land_area_m2,frontage_m,ownership_fraction,built_area_m2,property_tax_use_code,property_tax_use_description,property_tax_standard_code,property_tax_standard_description,property_tax_adjusted_construction_year
0,01700600397,R ARAGUAIA,744,<NA>,PARI,<NA>,03034000,1.Compra e venda,1200000.0,2005-09-05 00:00:00,...,Ativo Predial,3145,91.85,1.0,2364,051,ARMAZÉNS GERAIS E DEPÓSITOS,84,"OFICINA, POSTO DE SERVIÇO, ARMAZÉM/DEPÓSITO, I...",1950.0
1,00404805434,R ALEXANDRE LEVI,100,AP 04 E VG,<NA>,ED PQ CANTAREIRA,01520000,1.Compra e venda,82131.35,2005-10-07 00:00:00,...,Ativo Predial,7774,95.69,0.0032,93,020,APARTAMENTO EM CONDOMÍNIO (EXIGE FRAÇÃO IDEAL),22,RESIDENCIAL VERTICAL,2002.0
2,02003700455,R CAMARAJIBE,237,241,BARRA FUNDA,<NA>,01154050,1.Compra e venda,37423.0,2005-12-20 00:00:00,...,Ativo Predial,453,30.19,1.0,656,040,LOJA,32,COMERCIAL HORIZONTAL,1998.0
3,10133100153,R STA BIBIANA,178,<NA>,BUTANTA,<NA>,05627030,3.Adjudicação (vinculada a processo de execuçã...,93748.0,2004-11-03 00:00:00,...,Ativo Territorial,340,10.0,1.0,0,000,TERRENO,00,TERRENO,NaN
4,08012900947,R DARDANELOS,163,<NA>,<NA>,<NA>,05468010,1.Compra e venda,110000.0,2005-11-24 00:00:00,...,Ativo Predial,128,3.4,1.0,89,010,RESIDÊNCIA,12,RESIDENCIAL HORIZONTAL,1973.0


In [8]:
df.dtypes

property_tax_id                            string
street_name                                string
street_number                              string
address_complement                         string
neighborhood                               string
address_reference                          string
postal_code                                string
transaction_nature                         string
declared_transaction_value                 object
transaction_date                           object
reference_market_value                     object
transferred_share_pct                      object
proportional_reference_market_value        object
tax_calculation_base                       object
financing_type                             string
financed_amount                            object
registry_office                            string
property_registration_id                   string
property_tax_status                        string
land_area_m2                               object


In [9]:
extra_values

,file,sheet,excel_row,excel_column,value
0,_data/itbi/2008.xlsx,MAR-2008,7321,29,1971
1,_data/itbi/2011.xlsx,JUL-2011,6632,29,1971
2,_data/itbi/2013.xlsx,FEV-2013,7370,29,1989
3,_data/itbi/2014.xlsx,ABR-2014,8669,29,1989
4,_data/itbi/2023.xlsx,OUT-2023,11929,29,2003
5,_data/itbi/2025.xlsx,ABR-2025,15119,29,1986
6,_data/itbi/2025.xlsx,JUN-2025,4476,29,1999
7,_data/itbi/2025.xlsx,JUN-2025,4480,29,1999
8,_data/itbi/2025.xlsx,SET-2025,7023,29,1956
9,_data/itbi/2025.xlsx,NOV-2025,9442,29,1970


## Enrich transactions with monthly BCB indicators

Load each economic series once over the transaction date range, consolidate by month, and match every transaction to its calendar month. Selic columns contain arithmetic means of published daily observations. Current-month means cover observations available so far; unpublished financing months and invalid transaction dates retain missing values.

In [10]:
monthly_indicators = load_or_fetch_monthly_indicators(
    df,
    CACHE_DIR / "monthly_economic_indicators.csv",
)
monthly_indicators

,transaction_month,selic_target_annual_pct_monthly_mean,selic_effective_daily_pct_monthly_mean,real_estate_financing_brl_millions
0,1986-06,NaN,0.066482,NaN
1,1986-07,NaN,0.083826,NaN
2,1986-08,NaN,0.120788,NaN
3,1986-09,NaN,0.131875,NaN
4,1986-10,NaN,0.084264,NaN
...,...,...,...,...
479,2026-05,14.500000,0.053400,23566.0
480,2026-06,14.391667,0.053028,26575.0
481,2026-07,14.250000,0.052531,28162.0
482,2026-08,14.040323,0.051784,25274.0


In [11]:
monthly_indicators

,transaction_month,selic_target_annual_pct_monthly_mean,selic_effective_daily_pct_monthly_mean,real_estate_financing_brl_millions
0,1986-06,NaN,0.066482,NaN
1,1986-07,NaN,0.083826,NaN
2,1986-08,NaN,0.120788,NaN
3,1986-09,NaN,0.131875,NaN
4,1986-10,NaN,0.084264,NaN
...,...,...,...,...
479,2026-05,14.500000,0.053400,23566.0
480,2026-06,14.391667,0.053028,26575.0
481,2026-07,14.250000,0.052531,28162.0
482,2026-08,14.040323,0.051784,25274.0


In [12]:
df = enrich_transactions(df, monthly_indicators)
df.head()

,property_tax_id,street_name,street_number,address_complement,neighborhood,address_reference,postal_code,transaction_nature,declared_transaction_value,transaction_date,...,built_area_m2,property_tax_use_code,property_tax_use_description,property_tax_standard_code,property_tax_standard_description,property_tax_adjusted_construction_year,transaction_month,selic_target_annual_pct_monthly_mean,selic_effective_daily_pct_monthly_mean,real_estate_financing_brl_millions
0,01700600397,R ARAGUAIA,744,<NA>,PARI,<NA>,03034000,1.Compra e venda,1200000.0,2005-09-05 00:00:00,...,2364,051,ARMAZÉNS GERAIS E DEPÓSITOS,84,"OFICINA, POSTO DE SERVIÇO, ARMAZÉM/DEPÓSITO, I...",1950.0,2005-09,19.616667,0.071071,NaN
1,00404805434,R ALEXANDRE LEVI,100,AP 04 E VG,<NA>,ED PQ CANTAREIRA,01520000,1.Compra e venda,82131.35,2005-10-07 00:00:00,...,93,020,APARTAMENTO EM CONDOMÍNIO (EXIGE FRAÇÃO IDEAL),22,RESIDENCIAL VERTICAL,2002.0,2005-10,19.306452,0.069892,NaN
2,02003700455,R CAMARAJIBE,237,241,BARRA FUNDA,<NA>,01154050,1.Compra e venda,37423.0,2005-12-20 00:00:00,...,656,040,LOJA,32,COMERCIAL HORIZONTAL,1998.0,2005-12,18.225806,0.066514,NaN
3,10133100153,R STA BIBIANA,178,<NA>,BUTANTA,<NA>,05627030,3.Adjudicação (vinculada a processo de execuçã...,93748.0,2004-11-03 00:00:00,...,0,000,TERRENO,00,TERRENO,NaN,2004-11,16.966667,0.062181,NaN
4,08012900947,R DARDANELOS,163,<NA>,<NA>,<NA>,05468010,1.Compra e venda,110000.0,2005-11-24 00:00:00,...,89,010,RESIDÊNCIA,12,RESIDENCIAL HORIZONTAL,1973.0,2005-11,18.883333,0.068603,NaN


## Geocode unique addresses

Results are persisted in SQLite and reused across kernel restarts. Requests are processed as they complete; service errors are retried on the next run.

In [13]:
geocoding_config = GeocodingConfig(
    max_workers=8,
    geohash_precision=5,
    cache_path=CACHE_DIR / "geocoding.sqlite3",
    cache_batch_size=100,
)

In [14]:
geocoder = GeocodingClient(geocoding_config)

In [15]:
df = geocoder.geocode(df)

Geocoding unique addresses: 100%|##########| 586463/586463 [00:00<?, ?it/s]

In [16]:
df[["street_name", "street_number", "latitude", "longitude", "geohash", "geocoding_status"]].head()

,street_name,street_number,latitude,longitude,geohash,geocoding_status
0,R ARAGUAIA,744,-23.523126,-46.616900,6gyf5,found
1,R ALEXANDRE LEVI,100,-23.558336,-46.620443,6gycg,found
2,R CAMARAJIBE,237,NaN,NaN,<NA>,not_found
3,R STA BIBIANA,178,NaN,NaN,<NA>,not_found
4,R DARDANELOS,163,-23.537025,-46.715562,6gyf0,found


## Explore and export

Preview purchases separately. The export includes all geocoded transactions, matching the original workflow.

In [17]:
geocoded_transactions = select_transactions(df, geocoded_only=True)

In [18]:
geocoded_transactions.head()

,property_tax_id,street_name,street_number,address_complement,neighborhood,address_reference,postal_code,transaction_nature,declared_transaction_value,transaction_date,...,transaction_month,selic_target_annual_pct_monthly_mean,selic_effective_daily_pct_monthly_mean,real_estate_financing_brl_millions,geocoding_address,latitude,longitude,geocoding_status,geocoding_error,geohash
0,01700600397,R ARAGUAIA,744,<NA>,PARI,<NA>,03034000,1.Compra e venda,1200000.0,2005-09-05 00:00:00,...,2005-09,19.616667,0.071071,NaN,"R ARAGUAIA, 744, PARI, 03034000, SÃO PAULO, SP...",-23.523126,-46.616900,found,<NA>,6gyf5
1,00404805434,R ALEXANDRE LEVI,100,AP 04 E VG,<NA>,ED PQ CANTAREIRA,01520000,1.Compra e venda,82131.35,2005-10-07 00:00:00,...,2005-10,19.306452,0.069892,NaN,"R ALEXANDRE LEVI, 100, 01520000, SÃO PAULO, SP...",-23.558336,-46.620443,found,<NA>,6gycg
4,08012900947,R DARDANELOS,163,<NA>,<NA>,<NA>,05468010,1.Compra e venda,110000.0,2005-11-24 00:00:00,...,2005-11,18.883333,0.068603,NaN,"R DARDANELOS, 163, 05468010, SÃO PAULO, SP, BR...",-23.537025,-46.715562,found,<NA>,6gyf0
6,03806909393,R VERGUEIRO,2045,SALA 802 E V,<NA>,EDIF PAULISTA TOWER,04101000,1.Compra e venda,73708.6,2005-12-29 00:00:00,...,2005-12,18.225806,0.066514,NaN,"R VERGUEIRO, 2045, 04101000, SÃO PAULO, SP, BR...",-23.579985,-46.639110,found,<NA>,6gycf
7,08510201587,R BARTOLOMEU FEIO,122,AP 14,<NA>,ED GOLD STAR,04580000,1.Compra e venda,60000.0,2005-12-30 00:00:00,...,2005-12,18.225806,0.066514,NaN,"R BARTOLOMEU FEIO, 122, 04580000, SÃO PAULO, S...",-23.617332,-46.687959,found,<NA>,6gyc9


In [19]:
purchase_transactions = select_transactions(geocoded_transactions, purchases_only=True)

In [20]:
purchase_transactions.head()

,property_tax_id,street_name,street_number,address_complement,neighborhood,address_reference,postal_code,transaction_nature,declared_transaction_value,transaction_date,...,transaction_month,selic_target_annual_pct_monthly_mean,selic_effective_daily_pct_monthly_mean,real_estate_financing_brl_millions,geocoding_address,latitude,longitude,geocoding_status,geocoding_error,geohash
0,01700600397,R ARAGUAIA,744,<NA>,PARI,<NA>,03034000,1.Compra e venda,1200000.0,2005-09-05 00:00:00,...,2005-09,19.616667,0.071071,NaN,"R ARAGUAIA, 744, PARI, 03034000, SÃO PAULO, SP...",-23.523126,-46.616900,found,<NA>,6gyf5
1,00404805434,R ALEXANDRE LEVI,100,AP 04 E VG,<NA>,ED PQ CANTAREIRA,01520000,1.Compra e venda,82131.35,2005-10-07 00:00:00,...,2005-10,19.306452,0.069892,NaN,"R ALEXANDRE LEVI, 100, 01520000, SÃO PAULO, SP...",-23.558336,-46.620443,found,<NA>,6gycg
4,08012900947,R DARDANELOS,163,<NA>,<NA>,<NA>,05468010,1.Compra e venda,110000.0,2005-11-24 00:00:00,...,2005-11,18.883333,0.068603,NaN,"R DARDANELOS, 163, 05468010, SÃO PAULO, SP, BR...",-23.537025,-46.715562,found,<NA>,6gyf0
6,03806909393,R VERGUEIRO,2045,SALA 802 E V,<NA>,EDIF PAULISTA TOWER,04101000,1.Compra e venda,73708.6,2005-12-29 00:00:00,...,2005-12,18.225806,0.066514,NaN,"R VERGUEIRO, 2045, 04101000, SÃO PAULO, SP, BR...",-23.579985,-46.639110,found,<NA>,6gycf
7,08510201587,R BARTOLOMEU FEIO,122,AP 14,<NA>,ED GOLD STAR,04580000,1.Compra e venda,60000.0,2005-12-30 00:00:00,...,2005-12,18.225806,0.066514,NaN,"R BARTOLOMEU FEIO, 122, 04580000, SÃO PAULO, S...",-23.617332,-46.687959,found,<NA>,6gyc9


In [21]:
geocoded_transactions.to_csv(OUTPUT_PATH)